## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
聯絡
vincentman1027@gmail.com
www.linkedin.com/in/vincent-man-
b73841167 (LinkedIn)
熱門技能
Vue.js
Information Security
Python (Programming Language)
Certifications
AWS Certified AI Practitioner
Vincent Man
System Analyst
香港特別行政區
經歷
Peoplebank
Contract System Analyst (Landsd Department)
2024 年 3 月 - Present (2 年 6 個月)
Island
Administer ArcGIS Server, managing Map and GeoProcessing Services.
• Lead data analysis using ArcGIS Desktop with a focus on customized
ArcMap applications. 
• Perform full-stack enhancements and bug fixing using VUE3, JQuery, and
C#.
• Develop Python scripts for automated data conversion and optimize Oracle
stored procedures.
• Act as a technical consultant to users and manage vendor negotiations.
Bank of China
Contract System Analyst  
2023 年 10 月 - 2024 年 3 月 (6 個月)
離島區
Spearheaded a revamp application using Nestjs.
• Provided production support for Java (Struts 2) and DB2 database systems.
• Collaborated with stakeholders to collect and document user requirement

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Vincent Man"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Vincent Man. You are answering questions on Vincent Man's website, particularly questions related to Vincent Man's career, background, skills and experience. Your responsibility is to represent Vincent Man for interactions on the website as faithfully as possible. You are given a summary of Vincent Man's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\n\xa0 \xa0\n聯絡\nvincent

In [11]:
# Option A: DeepSeek R1 (Reasoning)
model_DEEPSEEK_R1 = "deepseek/deepseek-r1"

# Option B: DeepSeek V3 (Fast & Powerful General Model)
model_DEEPSEEK_CHAT = "deepseek/deepseek-chat"

# Option C: Qwen 2.5 72B (Excellent Chinese/English Support)
model_QWEN = "qwen/qwen-2.5-72b-instruct"

#Optgion D: Grok 4.6 (Fast & Powerful General Model)    
model_GROK = "x-ai/grok-4.6"



In [14]:
import os
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model_DEEPSEEK_CHAT, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [16]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [17]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [18]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [19]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [24]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.chat.completions.parse(model=model_DEEPSEEK_R1, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [25]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model=model_DEEPSEEK_CHAT, messages=messages)
reply = response.choices[0].message.content

In [26]:
reply

"No, I do not currently hold any patents. While I have extensive experience in system analysis, software development, and technical consulting across various industries (particularly in banking, telecommunications, and government sectors), my work has focused more on practical application development, system enhancements, and technical solutions rather than patentable inventions.\n\nMy expertise includes full-stack development (Vue.js, Java, .NET Core), geospatial systems (ArcGIS), and system integration - all areas where patents are less common compared to utility or design patents. That said, I'm always exploring innovative solutions in these domains where patent opportunities might arise in the future.\n\nWould you like to discuss any particular technical area or innovation you're interested in? I'd be happy to share more about my specific technical experiences."

In [27]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The Agent's response is acceptable. \n\nThe response truthfully acknowledges that Vincent Man does not hold any patents, which is consistent with the provided context—there is no mention of patents in his LinkedIn profile or summary. \n\nThe explanation effectively frames his focus on practical applications and technical solutions in fields where patents are less common, appropriately aligning with his documented experience in system analysis, software development, and technical consulting in banking, telecommunications, and government domains. \n\nThe tone remains professional and engaging, maintaining alignment with his role (as a representative for professional engagements) by proactively offering further detail on technical areas, which is a constructive way to shift focus to his strengths. \n\nNo inaccuracies or hallucinations are present.")

In [28]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [29]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model= model_DEEPSEEK_CHAT, messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [30]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
